# Bank Marketing — Data Preprocessing

## Objective

This notebook prepares the Bank Marketing dataset for machine
learning. It defines the prediction target, prevents data leakage,
creates training and test sets, and builds reproducible preprocessing
transformations for numerical and categorical features.

In [23]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [5]:
df = pd.read_csv(
    "../data/raw/bank-full.csv",
    sep=";",
)

In [6]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [7]:
X = df.drop(columns=["y", "duration"]).copy()

In [8]:
y = df["y"].map({
    "yes": 1,
    "no": 0,
})

In [9]:
y.value_counts()

y
0    39922
1     5289
Name: count, dtype: int64

In [10]:
y.isnull().sum()

np.int64(0)

In [11]:
X["previously_contacted"] = (
    X["pdays"] != -1
).astype("int8")

In [12]:
X["pdays"] = X["pdays"].replace(-1, 0)

In [13]:
X[
    ["pdays", "previously_contacted"]
].value_counts().head(10)

pdays  previously_contacted
0      0                       36954
182    1                         167
92     1                         147
91     1                         126
183    1                         126
181    1                         117
370    1                          99
184    1                          85
364    1                          77
95     1                          74
Name: count, dtype: int64

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [15]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (36168, 16)
X_test: (9043, 16)


In [16]:
pd.DataFrame({
    "training": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
})

,training,test
y,,
0,0.883018,0.883003
1,0.116982,0.116997


## Preprocessing Pipeline

In [28]:
categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_features = [
    "age",
    "balance",
    "day",
    "campaign",
    "pdays",
    "previous",
]

binary_features = [
    "previously_contacted",
]

In [29]:
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)
print("Binary features:", binary_features)

Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Numeric features: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
Binary features: ['previously_contacted']


In [30]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

In [31]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)

In [32]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features,
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ]
)

In [36]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [37]:
print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (36168, 51)
Processed test shape: (9043, 51)


In [38]:
feature_names = preprocessor.get_feature_names_out()

print("Number of output features:", len(feature_names))
feature_names

Number of output features: 51


array(['numeric__age', 'numeric__balance', 'numeric__day',
       'numeric__campaign', 'numeric__pdays', 'numeric__previous',
       'categorical__job_admin.', 'categorical__job_blue-collar',
       'categorical__job_entrepreneur', 'categorical__job_housemaid',
       'categorical__job_management', 'categorical__job_retired',
       'categorical__job_self-employed', 'categorical__job_services',
       'categorical__job_student', 'categorical__job_technician',
       'categorical__job_unemployed', 'categorical__job_unknown',
       'categorical__marital_divorced', 'categorical__marital_married',
       'categorical__marital_single', 'categorical__education_primary',
       'categorical__education_secondary',
       'categorical__education_tertiary',
       'categorical__education_unknown', 'categorical__default_no',
       'categorical__default_yes', 'categorical__housing_no',
       'categorical__housing_yes', 'categorical__loan_no',
       'categorical__loan_yes', 'categorical__contac

## Preprocessing Validation

In [41]:
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

assert X_train.index.intersection(X_test.index).empty

assert X_train_processed.shape[0] == len(y_train)
assert X_test_processed.shape[0] == len(y_test)

assert (
    X_train_processed.shape[1]
    == X_test_processed.shape[1]
    == len(feature_names)
)

assert "duration" not in X_train.columns
assert set(y_train.unique()) == {0, 1}

print("All preprocessing checks passed.")

All preprocessing checks passed.


## Preprocessing Summary

- The target variable was encoded as `0` for `no` and `1` for `yes`.
- The `duration` feature was excluded to prevent data leakage in a pre-contact prediction scenario.
- A binary feature named `previously_contacted` was created before replacing the special value `pdays = -1` with zero.
- The data was divided into stratified training and test sets, preserving the original target distribution.
- Numerical features are imputed using their training-set medians and standardized.
- Categorical features are imputed using their most frequent training-set values and transformed using one-hot encoding.
- Existing `unknown` values are retained as explicit categories.
- Binary engineered features are passed through without scaling.
- The preprocessing pipeline is fitted only on the training set and produces 51 model-ready features.